## Evaluation Framework

In order to set up an evaluation, we need to consider what we are aiming to get. Down the line, the ESCO nodes found by our vector search will be used to build a prompt so that the Large Language Model can ask users to validate a number of skills and occupations found through vector search. For that reason, we don't want to focus so much on the precision (that is, reducing the amount of false positives), but rather on the recall (that is, reducing the amount of false negatives). 

On this premise, we design an evaluation method (accuracy@k) in which we consider a success (a score of 1) if the correct node is found within the top k classes and an insuccess (a score of 0) otherwise. This method could be further refined by considering the score to be inversely proportional to the rank in which the correct node is found. However, since we're not concerned with the rank, we will use an averaged 0-1 score.

In [1]:
from typing import List

def accuracy_within_list(prediction: List[List[str]], true: List[str]):
    """Calculates the average accuracy considering
    for each prediction a value of 1 if the correct
    node is in the predicted list and 0 otherwise.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[str]): list of the true nodes in the
            dataset.

    Returns:
        float: average accuracy over all the test set.
    """
    assert len(prediction) == len(true)
    total_correct = 0
    for pred_list, true_val in zip(prediction, true):
        total_correct+=int(true_val in pred_list)
    return total_correct/len(true)

def accuracy_at_k(prediction: List[List[str]], true: List[str], k: int):
    """Calculates the average accuracy considering
    for each prediction a value of 1 if the correct
    node is in the top k elements of the predicted list
    and 0 otherwise.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[str]): list of the true nodes in the
            dataset.
        k (int): number of elements of each list to be considered.

    Returns:
        float: average accuracy over all the test set.
    """
    assert len(prediction) == len(true)
    total_correct = 0
    for pred_list, true_val in zip(prediction, true):
        total_correct+=int(true_val in pred_list[:k])
    return total_correct/len(true)

## Evaluation goals

The objective of the evaluation is to choose which model and hyperparameters have the highest accuracy at k. For a given embedding model, the hyperparameters are the following:

1. **How to embed a node of the graph**: which combination of the fields guarantees the best performance when embedded.
2. **Score function**: which function should be used to retrieve the most similar nodes (cosine, l2 distance or scalar product).
3. **Using Maximal Marginal Relevance**: whether we should use MMR to get more diverse results.

We will use ChromaDB as a local vector store and get the ESCO data from a local csv file. We will restrict our evaluation to the gecko003 model, but this can be repeated with any other model.

In [2]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv("/Users/francescopreta/coding/compass/backend/.env")

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
REPO_ID = "tabiya/hahu_test"
FILENAME = "redacted_hahu_test_with_id.csv"

df_test = pd.read_csv(
    hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset", token=HF_TOKEN)
)

OCCUPATION_CSV_PATH = "/Users/francescopreta/coding/compass/poc/data/occupations.csv"
df_database = pd.read_csv(OCCUPATION_CSV_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

/Users/francescopreta/miniconda3/envs/backend/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In what follows, we will pre-compute the strings and the corresponding embeddings using the Gecko model. We will use manual batching to speed up the process, as the vertex API doesn't support batching and fails if the list length is larger than 250 elements or the sum of tokens is higher than 20.000.

In [15]:
def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

We define the functions generating the string that can be embedded from each document and save their computation in a dictionary. Then we save the corresponding embeddings in another dictionary, as well as the embeddings of the test set, that will be used for query retrieval.

In [17]:
# Functions defining strings to embed
def description(df):
    return df["DESCRIPTION"]

def preferred_label(df):
    return df["PREFERREDLABEL"]

def all_occupations(df):
    return f"""Occupation Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Occupation Description: {df['DESCRIPTION']}"""

def label_and_description(df):
    return f"{df['PREFERREDLABEL']}\n{df['DESCRIPTION']}"

function_to_method ={"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_OCCUPATIONS":all_occupations, "LABEL_AND_DESCRIPTION": label_and_description}

In [ ]:
# Embedding precomputation
method_to_embeddings = {}
method_to_documents = {}
for method in function_to_method:
    method_to_documents[method] = list(df_database.progress_apply(function_to_method[method], axis=1))
    method_to_embeddings[method] = embed_strings_in_batch(method_to_documents[method], model)

test_embeddings = embed_strings_in_batch(list(df_test["synthetic_query"]), model)

In [5]:
# Functions "maximal_marginal_relevance" and "cosine_similarity"
# are duplicated respectively from modules:
#    - "libs/community/langchain_community/vectorstores/utils.py"
#    - "libs/community/langchain_community/utils/math.py"
from typing import List, Union

import numpy as np

Matrix = Union[List[List[float]], List[np.ndarray], np.ndarray]


def cosine_similarity(X: Matrix, Y: Matrix) -> np.ndarray:
    """Row-wise cosine similarity between two equal-width matrices."""
    if len(X) == 0 or len(Y) == 0:
        return np.array([])

    X = np.array(X)
    Y = np.array(Y)
    if X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Number of columns in X and Y must be the same. X has shape {X.shape} "
            f"and Y has shape {Y.shape}."
        )
    X_norm = np.linalg.norm(X, axis=1)
    Y_norm = np.linalg.norm(Y, axis=1)
    # Ignore divide by zero errors run time warnings as those are handled below.
    with np.errstate(divide="ignore", invalid="ignore"):
        similarity = np.dot(X, Y.T) / np.outer(X_norm, Y_norm)
    similarity[np.isnan(similarity) | np.isinf(similarity)] = 0.0
    return similarity



def maximal_marginal_relevance(
    query_embedding: np.ndarray,
    embedding_list: list,
    lambda_mult: float = 0.5,
    k: int = 10,
) -> List[int]:
    """Calculate maximal marginal relevance."""
    if min(k, len(embedding_list)) <= 0:
        return []
    if query_embedding.ndim == 1:
        query_embedding = np.expand_dims(query_embedding, axis=0)
    similarity_to_query = cosine_similarity(query_embedding, embedding_list)[0]
    most_similar = int(np.argmax(similarity_to_query))
    idxs = [most_similar]
    selected = np.array([embedding_list[most_similar]])
    while len(idxs) < min(k, len(embedding_list)):
        best_score = -np.inf
        idx_to_add = -1
        similarity_to_selected = cosine_similarity(embedding_list, selected)
        for i, query_score in enumerate(similarity_to_query):
            if i in idxs:
                continue
            redundant_score = max(similarity_to_selected[i])
            equation_score = (
                lambda_mult * query_score - (1 - lambda_mult) * redundant_score
            )
            if equation_score > best_score:
                best_score = equation_score
                idx_to_add = i
        idxs.append(idx_to_add)
        selected = np.append(selected, [embedding_list[idx_to_add]], axis=0)
    return idxs


We create multiple chromadb collections to store our data in memory with different embeddings depending on the method used and on the function used for querying. On these, we save the Occupation ESCO database with all their metadatas.

In [6]:
import chromadb

client = chromadb.Client()
embedding_methods = ["DESCRIPTION", "PREFERREDLABEL", "ALL_OCCUPATIONS", "LABEL_AND_DESCRIPTION"]
score_function = ["cosine", "l2", "ip"]
for method in embedding_methods:
    for score in score_function:
        collection_name = f'{method}_{score}'
        collection_metadata = {"hnsw:space":score}
        collection = client.create_collection(name=collection_name, metadata=collection_metadata)
        collection.add(
            documents = method_to_documents[method],
            metadatas = df_database.to_dict('records'),
            embeddings = method_to_embeddings[method],
            ids = [f"id_{i}" for i in range(len(df_database))]
        )

Finally, we run the evaluation as follows:

1. We choose a score function and a method and load the corresponding collection.
2. For each element in the test set, we find the top 100 documents in the collection ordered by scoring rank.
3. We filter those documents by maximal marginal relevance to find the top 10 documents with this function.
4. We evaluate the accuracy on the top k for k=1,3,5,10 for the original retrieved documents.
5. We evaluate the accuracy on the top k for k=1,3,5,10 for the documents filtered by maximal marginal relevance.
6. We save the results in a dataframe to be analyzed for later use.

In [12]:
eval_data = []
for method in embedding_methods:
    for score in score_function:
        collection_name = f'{method}_{score}'
        # Fetch collection
        collection = client.get_collection(name=collection_name)
        # Initialize lists to save results
        vector_search_results = []
        mmr_vector_search_results = []
        for test_embedding in test_embeddings:
            # Find the top 100 documents and save them in vector_search_results
            documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
            vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
            # Find the top 10 documents according to MMR and save them in mmr_vector_search_results
            result_embeddings = [elem for elem in documents_from_search["embeddings"][0]]
            mmr_ids = maximal_marginal_relevance(np.array(test_embedding), result_embeddings)
            mmr_vector_search_results.append([elem["CODE"] for index, elem in enumerate(documents_from_search["metadatas"][0]) if index in mmr_ids])
        # Evaluate accuracy at k for k=1, 3, 5, 10
        for k in [1, 3, 5, 10]:
            acc_at_k = accuracy_at_k(vector_search_results, list(df_test["esco_code"]), k)
            eval_data.append({"method":method, "score function":score, "MMR": False, "k":k, "accuracy": acc_at_k})
            print(f"Method: {method}, Score function: {score}, MMR: False\n Accuracy at {k}: {round(acc_at_k,4)}")
            acc_at_k = accuracy_at_k(mmr_vector_search_results, list(df_test["esco_code"]), k)
            eval_data.append({"method":method, "score function":score, "MMR": True, "k":k, "accuracy": acc_at_k})
            print(f"Method: {method}, Score function: {score}, MMR: True\n Accuracy at {k}: {round(acc_at_k,4)}")
# Save the results in a dataframe
eval_df = pd.DataFrame(eval_data)

Method: DESCRIPTION, Score function: cosine, MMR: False
 Accuracy at 1: 0.3364
Method: DESCRIPTION, Score function: cosine, MMR: True
 Accuracy at 1: 0.3364
Method: DESCRIPTION, Score function: cosine, MMR: False
 Accuracy at 3: 0.5364
Method: DESCRIPTION, Score function: cosine, MMR: True
 Accuracy at 3: 0.38
Method: DESCRIPTION, Score function: cosine, MMR: False
 Accuracy at 5: 0.6109
Method: DESCRIPTION, Score function: cosine, MMR: True
 Accuracy at 5: 0.3909
Method: DESCRIPTION, Score function: cosine, MMR: False
 Accuracy at 10: 0.7036
Method: DESCRIPTION, Score function: cosine, MMR: True
 Accuracy at 10: 0.3945
Method: DESCRIPTION, Score function: l2, MMR: False
 Accuracy at 1: 0.3364
Method: DESCRIPTION, Score function: l2, MMR: True
 Accuracy at 1: 0.3364
Method: DESCRIPTION, Score function: l2, MMR: False
 Accuracy at 3: 0.5364
Method: DESCRIPTION, Score function: l2, MMR: True
 Accuracy at 3: 0.38
Method: DESCRIPTION, Score function: l2, MMR: False
 Accuracy at 5: 0.6109
M

In [13]:
# Save the evaluation results locally
OUTPUT_EVAL_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/hahu_jobs_eval_results.csv"
eval_df.to_csv(OUTPUT_EVAL_PATH)

The evaluation results are as follows:

|   | method             | score function | MMR   | k  | accuracy            |
|---|--------------------|----------------|-------|----|---------------------|
| 46| PREFERREDLABEL     | ip             | False | 10 | 0.7345454545454545  |
| 30| PREFERREDLABEL     | cosine         | False | 10 | 0.7345454545454545  |
| 14| DESCRIPTION        | l2             | False | 10 | 0.7036363636363636  |
| 6 | DESCRIPTION        | cosine         | False | 10 | 0.7036363636363636  |
| 78| LABEL_AND_DESCRIPTION | cosine     | False | 10 | 0.7090909090909091  |
| 62| ALL_OCCUPATIONS    | l2             | False | 10 | 0.7272727272727273  |
| 54| ALL_OCCUPATIONS    | cosine         | False | 10 | 0.7272727272727273  |
| 38| PREFERREDLABEL     | l2             | False | 10 | 0.7345454545454545  |
| 22| DESCRIPTION        | ip             | False | 10 | 0.7036363636363636  |
| 70| ALL_OCCUPATIONS    | ip             | False | 10 | 0.7272727272727273  |
| 94| LABEL_AND_DESCRIPTION | ip         | False | 10 | 0.7090909090909091  |
| 86| LABEL_AND_DESCRIPTION | l2         | False | 10 | 0.7090909090909091  |
| 58| ALL_OCCUPATIONS    | l2             | False | 5  | 0.6509090909090909  |
| 52| ALL_OCCUPATIONS    | cosine         | False | 5  | 0.6509090909090909  |
| 44| PREFERREDLABEL     | ip             | False | 5  | 0.6527272727272727  |
| 28| PREFERREDLABEL     | cosine         | False | 5  | 0.6527272727272727  |
| 76| LABEL_AND_DESCRIPTION | cosine     | False | 5  | 0.6054545454545455  |
| 60| ALL_OCCUPATIONS    | l2             | False | 5  | 0.6509090909090909  |
| 56| ALL_OCCUPATIONS    | l2             | False | 1  | 0.3418181818181818  |
| 48| ALL_OCCUPATIONS    | cosine         | False | 1  | 0.3418181818181818  |
| 40| PREFERREDLABEL     | ip             | False | 1  | 0.37454545454545457 |
| 24| PREFERREDLABEL     | cosine         | False | 1  | 0.37454545454545457 |
| 80| LABEL_AND_DESCRIPTION | l2         | False | 1  | 0.36727272727272725 |
| 72| LABEL_AND_DESCRIPTION | cosine     | False | 1  | 0.36727272727272725 |
| 16| DESCRIPTION        | ip             | False | 1  | 0.33636363636363636 |
| 0 | DESCRIPTION        | cosine         | False | 1  | 0.33636363636363636 |
| 64| ALL_OCCUPATIONS    | ip             | False | 1  | 0.3418181818181818  |
| 32| PREFERREDLABEL     | l2             | False | 1  | 0.37454545454545457 |
| 8 | DESCRIPTION        | l2             | False | 1  | 0.33636363636363636 |
| 88| LABEL_AND_DESCRIPTION | ip         | False | 1  | 0.36727272727272725 |
| 1 | DESCRIPTION        | cosine         | True  | 1  | 0.33636363636363636 |
| 25| PREFERREDLABEL     | cosine         | True  | 1  | 0.37454545454545457 |
| 9 | DESCRIPTION        | l2             | True  | 1  | 0.33636363636363636 |
| 41| PREFERREDLABEL     | ip             | True  | 1  | 0.37454545454545457 |
| 17| DESCRIPTION        | ip             | True  | 1  | 0.33636363636363636 |
| 33| PREFERREDLABEL     | l2             | True  | 1  | 0.37454545454545457 |
| 89| LABEL_AND_DESCRIPTION | ip         | True  | 1  | 0.36727272727272725 |
| 49| ALL_OCCUPATIONS    | cosine         | True  | 1  | 0.3418181818181818  |
| 65| ALL_OCCUPATIONS    | ip             | True  | 1  | 0.3418181818181818  |
| 97| LABEL_AND_DESCRIPTION | ip         | True  | 10 | 0.4254545454545455  |
| 93| LABEL_AND_DESCRIPTION | ip         | True  | 5  | 0.4254545454545455  |
| 71| ALL_OCCUPATIONS    | ip             | True  | 10 | 0.42727272727272725 |
| 55| ALL_OCCUPATIONS    | cosine         | True  | 10 | 0.42727272727272725 |
| 47| PREFERREDLABEL     | ip             | True  | 10 | 0.44181818181818183 |
| 31| PREFERREDLABEL     | cosine         | True  | 10 | 0.44181818181818183 |
| 37| PREFERREDLABEL     | l2             | True  | 5  | 0.44                |
| 29| PREFERREDLABEL     | cosine         | True  | 5  | 0.44                |
| 53| ALL_OCCUPATIONS    | cosine         | True  | 5  | 0.41818181818181815 |
| 69| ALL_OCCUPATIONS    | ip             | True  | 5  | 0.41818181818181815 |
| 21| DESCRIPTION        | ip             | True  | 5  | 0.39090909090909093 |
| 5 | DESCRIPTION        | cosine         | True  | 5  | 0.39090909090909093 |
| 93| LABEL_AND_DESCRIPTION | ip         | True  | 5  | 0.4254545454545455  |
| 61| ALL_OCCUPATIONS    | l2             | True  | 5  | 0.41818181818181815 |
| 17| DESCRIPTION        | ip             | True  | 5  | 0.39090909090909093 |
| 45| PREFERREDLABEL     | ip             | True  | 5  | 0.44                |
| 29| PREFERREDLABEL     | cosine         | True  | 5  | 0.44                |
| 77| LABEL_AND_DESCRIPTION | cosine     | True  | 5  | 0.4254545454545455  |
| 13| DESCRIPTION        | l2             | True  | 5  | 0.39090909090909093 |
| 41| PREFERREDLABEL     | ip             | True  | 1  | 0.37454545454545457 |
| 25| PREFERREDLABEL     | cosine         | True  | 1  | 0.37454545454545457 |
| 85| LABEL_AND_DESCRIPTION | l2         | True  | 5  | 0.4254545454545455  |
| 49| ALL_OCCUPATIONS    | cosine         | True  | 1  | 0.3418181818181818  |
| 37| PREFERREDLABEL     | l2             | True  | 5  | 0.44                |
| 85| LABEL_AND_DESCRIPTION | l2         | True  | 5  | 0.4254545454545455  |
| 61| ALL_OCCUPATIONS    | l2             | True  | 5  | 0.41818181818181815 |
| 21| DESCRIPTION        | ip             | True  | 5  | 0.39090909090909093 |
| 5 | DESCRIPTION        | cosine         | True  | 5  | 0.39090909090909093 |
| 33| PREFERREDLABEL     | l2             | True  | 1  | 0.37454545454545457 |
| 57| ALL_OCCUPATIONS    | l2             | True  | 1  | 0.3418181818181818  |
| 69| ALL_OCCUPATIONS    | ip             | True  | 5  | 0.41818181818181815 |
| 13| DESCRIPTION        | l2             | True  | 5  | 0.39090909090909093 |
| 57| ALL_OCCUPATIONS    | l2             | True  | 1  | 0.3418181818181818  |
| 21| DESCRIPTION        | ip             | True  | 5  | 0.39090909090909093 |
| 45| PREFERREDLABEL     | ip             | True  | 5  | 0.44                |
| 77| LABEL_AND_DESCRIPTION | cosine     | True  | 5  | 0.4254545454545455  |
| 93| LABEL_AND_DESCRIPTION | ip         | True  | 5  | 0.4254545454545455  |
| 13| DESCRIPTION        | l2             | True  | 5  | 0.39090909090909093 |
| 29| PREFERREDLABEL     | cosine         | True  | 5  | 0.44                |
| 41| PREFERREDLABEL     | ip             | True  | 1  | 0.37454545454545457 |
| 25| PREFERREDLABEL     | cosine         | True  | 1  | 0.37454545454545457 |
| 85| LABEL_AND_DESCRIPTION | l2         | True  | 5  | 0.4254545454545455  |
| 49| ALL_OCCUPATIONS    | cosine         | True  | 1  | 0.3418181818181818  |
| 37| PREFERREDLABEL     | l2             | True  | 5  | 0.44                |
| 89| LABEL_AND_DESCRIPTION | ip         | True  | 1  | 0.36727272727272725 |
| 33| PREFERREDLABEL     | l2             | True  | 1  | 0.37454545454545457 |
| 57| ALL_OCCUPATIONS    | l2             | True  | 1  | 0.3418181818181818  |
| 69| ALL_OCCUPATIONS    | ip             | True  | 5  | 0.41818181818181815 |
| 21| DESCRIPTION        | ip             | True  | 5  | 0.39090909090909093 |
| 5 | DESCRIPTION        | cosine         | True  | 5  | 0.39090909090909093 |


The result of the evaluation are as follows:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods are either PREFERREDLABEL or ALL_OCCUPATIONS judging from the results from 5 and 10. We are choosing PREFERREDLABEL given that it has a small margin over ALL_OCCUPATIONS and it's easier to implement as its value is already present in the dataset.

We can now write a function to embed the test and database dataframes and obtain the evaluation results.

In [ ]:
# Best results are given without MMR and using only the preferred label for inputs
def evaluate(model: TextEmbeddingModel, database: pd.DataFrame, test_set: pd.DataFrame, k: int) -> float:
    """Returns the accuracy at k for the vector search through the embedding generated 
    by a given model on a provided dataset. The model must be a TextEmbeddingModel,
    the database must be a Dataframe containing the columns 'PREFERREDLABEL' and 'CODE',
    while the test set must be a Dataframe containing the columns 'synthetic_query' and
    'esco_code'.

    Args:
        model (TextEmbeddingModel): model to be evaluated.
        database (pd.DataFrame): database from which the data should be retrieved.
        test_set (pd.DataFrame): test set for the evaluation.
        k (int): k for the accuracy at k to be evaluated.

    Returns:
        float: accuracy at k
    """
    assert k<=100
    # Calculate embeddings for the database
    db_embeddings = embed_strings_in_batch(list(database['PREFERREDLABEL']), model)
    # Calculate embeddings for the test set
    test_embeddings = embed_strings_in_batch(list(test_set['synthetic_query']), model)
    # Load a collection from chromadb and saves the data
    client = chromadb.Client()
    collection = client.create_collection(name='eval')
    collection.add(
        documents = list(database['PREFERREDLABEL']),
        metadatas = [{"CODE": row["CODE"]} for _, row in database.iterrows()],
        embeddings = db_embeddings,
        ids = [f"id_{i}" for i in range(len(database))]
    )
    # Save the data retrieved from the test set
    vector_search_results = []
    for test_embedding in test_embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    # Evaluate accuracy at k 
    return accuracy_at_k(vector_search_results, list(test_set["esco_code"]), k)
